# Final metric-wise CASB family comparison

Compare G-mean, minority F1, minority precision, and minority recall separately. For each metric, average each method over folds/seeds within each dataset/classifier, then equally across datasets. The combined table equally averages SVM, KNN, RF and LR. A family win means the highest average among CASB, CASB-relaxed, CASB-edited, and CASB-ROS exceeds every baseline average for that metric. Different family variants may win different metrics; this is not one method winning them all, and no significance claim is made.

Baselines: SMOTE, BSMOTE, ROS, ADASYN. No resampling is shown as a reference and is not included in the four-baseline family-win rule. RUS, SMOTEENN and MCSMOTE are excluded. Comparisons use the same complete datasets for all methods; ADASYN failures are reported explicitly. A separate 44-dataset table excludes ADASYN.

The saved experiments retain their original training-only ratio selection using mean G-mean/F1; changing reporting does not retune models or select ratios on test results. Composite scores are not used to declare winners here. All datasets have been inspected during development. The Abalone inputs use one-hot encoding. Related dataset variants/folds are not independent replications. CASB-ROS duplicates sources from CASB's original 50/30/20 source pools; the other variants retain their existing generation rules.


RF and LR were added after examining their benchmark scores. GaussianNB results remain in progress.md. Classifier selection is post hoc; reported tests are exploratory, not independent confirmation.

In [1]:
import json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
methods=['original','smote','bsmote','ros','adasyn','casb','casb_relaxed','casb_edited','casb_ros']
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
metrics=['gmean','f1','precision','recall']
rows=[]
paths=sorted(Path('.').glob('[0-9][0-9]_*_CASB.ipynb'))
assert len(paths)==44
expected={(s,f,c) for s in [42,137] for f in range(1,6) for c in ['svm', 'knn', 'rf', 'lr']}
for path in paths:
    nb=json.loads(path.read_text(encoding='utf-8'));meta=nb['metadata']['current_comparison']
    assert meta['complete'] and len(meta['records'])==360
    assert all(c.get('execution_count') is not None for c in nb['cells'] if c['cell_type']=='code')
    assert not any(o.get('output_type')=='error' for c in nb['cells'] for o in c.get('outputs',[]))
    record=meta['records']
    for source,digest in meta['input_sha256'].items():
        source_path=Path(source)
        if not source_path.exists(): source_path=Path('output')/source_path
        assert hashlib.sha256(source_path.read_bytes()).hexdigest()==digest
    dataset=nb['metadata']['benchmark']['dataset']
    for method in methods:
        subset=[r for r in record if r['method']==method]
        assert len(subset)==40 and {(r['seed'],r['fold'],r['classifier']) for r in subset}==expected
    for r in record:
        assert r['dataset']==dataset
        if 'error' in r: continue
        trials=[t for t in r.get('validation',[]) if 'ba' in t]
        if trials: assert r['ratio']==max(trials,key=lambda t:(t['ba'],-t['ratio']))['ratio']
        tn,fp,fn,tp=r['confusion']
        assert abs(r['f1']-2*tp/max(1,2*tp+fp+fn))<1e-12
        assert abs(r['gmean']-np.sqrt(tp/max(1,tp+fn)*tn/max(1,tn+fp)))<1e-12
    rows.extend(record)
frame=pd.DataFrame(rows)
assert len(frame)==15840 and frame.dataset.nunique()==44
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
valid=frame[frame.error.isna()]
assert np.isfinite(valid[metrics]).all().all()
bad=sorted(set(errors.dataset))
matched=valid[~valid.dataset.isin(bad)]
means=matched.groupby(['dataset','classifier','method'])[metrics].mean()
ds=means.groupby(['dataset','method']).mean()
print('44 datasets; 15,840 attempted outer fits; failed:',len(errors))
print('Matched all-method datasets:',matched.dataset.nunique(),'/44; excluded datasets:',bad)
if len(errors):display(errors[['dataset','classifier','seed','fold','method','error']])


44 datasets; 15,840 attempted outer fits; failed: 52
Matched all-method datasets: 40 /44; excluded datasets: ['ecoli-0-1-3-7_vs_2-6', 'iris0', 'shuttle-c0-vs-c4', 'shuttle-c2-vs-c4']


,dataset,classifier,seed,fold,method,error
2992,ecoli-0-1-3-7_vs_2-6,svm,137,2,adasyn,Not any neigbours belong to the majority class...
3001,ecoli-0-1-3-7_vs_2-6,knn,137,2,adasyn,Not any neigbours belong to the majority class...
3136,ecoli-0-1-3-7_vs_2-6,rf,137,2,adasyn,RuntimeError: Not any neigbours belong to the ...
3145,ecoli-0-1-3-7_vs_2-6,lr,137,2,adasyn,RuntimeError: Not any neigbours belong to the ...
7978,iris0,svm,42,4,adasyn,Not any neigbours belong to the majority class...
7987,iris0,knn,42,4,adasyn,Not any neigbours belong to the majority class...
7996,iris0,svm,42,5,adasyn,Not any neigbours belong to the majority class...
8005,iris0,knn,42,5,adasyn,Not any neigbours belong to the majority class...
8050,iris0,svm,137,3,adasyn,Not any neigbours belong to the majority class...
8059,iris0,knn,137,3,adasyn,Not any neigbours belong to the majority class...


In [2]:
def metric_family_winners(averages, baselines, variants):
    result=[]
    for metric in metrics:
        ours=averages.loc[variants,metric]
        others=averages.loc[baselines,metric]
        winner=ours.idxmax();baseline=others.idxmax()
        delta=float(ours.max()-others.max())
        result.append({'metric':metric,'best_CASB_variant':winner,'CASB_average_percent':100*ours.max(),
            'strongest_baseline':baseline,'baseline_average_percent':100*others.max(),
            'difference_pp':100*delta,'family_result':'WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'})
    return pd.DataFrame(result)
aggregate=ds.groupby('method')[metrics].mean()
display(Markdown('## Combined SVM/KNN/RF/LR averages (%) - matched datasets'))
display((100*aggregate.reindex(methods)).round(3))
display(Markdown('## Family wins for each metric - combined averages'))
combined_winners=metric_family_winners(aggregate,core,family)
display(combined_winners.round(3))
for clf in ['svm', 'knn', 'rf', 'lr']:
    average=means.xs(clf,level='classifier').groupby('method')[metrics].mean()
    display(Markdown('## '+clf.upper()+': method averages (%)'))
    display((100*average.reindex(methods)).round(3))
    display(Markdown('### '+clf.upper()+': family comparison for each metric'))
    display(metric_family_winners(average,core,family).round(3))


## Combined SVM/KNN/RF/LR averages (%) - matched datasets

,gmean,f1,precision,recall
method,,,,
original,62.307,57.038,68.593,52.231
smote,81.354,65.704,62.261,75.618
bsmote,79.519,64.806,61.005,74.100
ros,80.922,65.668,62.595,75.138
adasyn,81.518,64.400,59.440,76.742
casb,80.329,66.056,63.676,73.976
casb_relaxed,81.060,65.908,62.669,75.096
casb_edited,82.411,65.084,59.368,78.720
casb_ros,80.172,65.959,63.657,73.950


## Family wins for each metric - combined averages

,metric,best_CASB_variant,CASB_average_percent,strongest_baseline,baseline_average_percent,difference_pp,family_result
0,gmean,casb_edited,82.411,adasyn,81.518,0.894,WIN
1,f1,casb,66.056,smote,65.704,0.351,WIN
2,precision,casb,63.676,ros,62.595,1.081,WIN
3,recall,casb_edited,78.720,adasyn,76.742,1.978,WIN


## SVM: method averages (%)

,gmean,f1,precision,recall
method,,,,
original,59.194,54.653,66.558,49.652
smote,81.619,66.608,63.378,75.738
bsmote,79.669,65.610,61.503,74.052
ros,82.620,66.951,63.344,76.906
adasyn,81.499,64.737,59.979,76.397
casb,80.484,67.276,65.240,74.435
casb_relaxed,81.376,67.108,64.257,75.203
casb_edited,82.638,66.102,61.197,78.510
casb_ros,80.964,67.444,65.313,75.003


... (10 outputs left out)

## LR: method averages (%)

,gmean,f1,precision,recall
method,,,,
original,59.700,53.745,67.216,48.167
smote,83.943,63.749,57.216,79.978
bsmote,82.540,63.157,56.079,78.533
ros,84.139,63.734,57.317,80.386
adasyn,84.084,62.879,55.337,80.997
casb,83.235,63.998,58.205,78.232
casb_relaxed,83.609,64.027,57.756,79.477
casb_edited,84.284,63.207,55.311,82.250
casb_ros,83.510,64.112,58.122,78.832


### LR: family comparison for each metric

,metric,best_CASB_variant,CASB_average_percent,strongest_baseline,baseline_average_percent,difference_pp,family_result
0,gmean,casb_edited,84.284,ros,84.139,0.145,WIN
1,f1,casb_ros,64.112,smote,63.749,0.363,WIN
2,precision,casb,58.205,ros,57.317,0.888,WIN
3,recall,casb_edited,82.250,adasyn,80.997,1.253,WIN


In [3]:
# Dataset-wise counts are distinct from winning the collection-wide average.
def dataset_metric_counts(table,baselines,variants):
    result=[]
    for metric in metrics:
        matrix=table[metric].unstack('method')
        delta=matrix[variants].max(axis=1)-matrix[baselines].max(axis=1)
        result.append({'metric':metric,'family_wins':int((delta>1e-12).sum()),'ties':int((delta.abs()<=1e-12).sum()),
            'losses':int((delta < -1e-12).sum()),'datasets':len(matrix)})
    return pd.DataFrame(result)
display(Markdown('## Dataset-wise metric wins (retrospective best variant per dataset/metric)'))
display(dataset_metric_counts(ds,core,family))
for candidate,baseline in [('casb','smote'),('casb_ros','ros')]:
    delta=ds.xs(candidate,level='method')-ds.xs(baseline,level='method')
    display(Markdown('## '+candidate+' versus '+baseline))
    display(pd.DataFrame({'average_difference_pp':100*delta.mean(),'dataset_wins':(delta>1e-12).sum(),
        'ties':(delta.abs()<=1e-12).sum(),'losses':(delta < -1e-12).sum()}).round(3))


## Dataset-wise metric wins (retrospective best variant per dataset/metric)

,metric,family_wins,ties,losses,datasets
0,gmean,24,0,16,40
1,f1,23,0,17,40
2,precision,21,0,19,40
3,recall,24,2,14,40


## casb versus smote

,average_difference_pp,dataset_wins,ties,losses
gmean,-1.025,17,0,23
f1,0.351,18,0,22
precision,1.415,21,1,18
recall,-1.642,11,1,28


## casb_ros versus ros

,average_difference_pp,dataset_wins,ties,losses
gmean,-0.750,20,0,20
f1,0.291,21,0,19
precision,1.062,23,0,17
recall,-1.188,16,4,20


In [4]:
display(Markdown('## All 44 datasets, excluding ADASYN (%)'))
full=valid[valid.method!='adasyn']
assert len(full)==14080
fullmeans=full.groupby(['dataset','classifier','method'])[metrics].mean()
fullds=fullmeans.groupby(['dataset','method']).mean()
fullaverage=fullds.groupby('method').mean()
display((100*fullaverage).round(3))
display(metric_family_winners(fullaverage,['smote','bsmote','ros'],family).round(3))
display(Markdown('## Per-dataset matched averages (%)'))
with pd.option_context('display.max_rows',400):display((100*ds).round(3))
print('Results are metric-wise. No composite winner is reported. progress.md is unchanged.')


## All 44 datasets, excluding ADASYN (%)

,gmean,f1,precision,recall
method,,,,
bsmote,80.665,66.944,63.544,75.579
casb,81.399,67.963,65.827,75.479
casb_edited,83.287,67.030,61.847,79.791
casb_relaxed,82.067,67.892,64.989,76.496
casb_ros,81.258,67.891,65.823,75.452
original,64.670,59.712,70.379,55.329
ros,81.863,67.322,64.501,76.476
smote,82.331,67.658,64.567,76.969


,metric,best_CASB_variant,CASB_average_percent,strongest_baseline,baseline_average_percent,difference_pp,family_result
0,gmean,casb_edited,83.287,smote,82.331,0.956,WIN
1,f1,casb,67.963,smote,67.658,0.305,WIN
2,precision,casb,65.827,smote,64.567,1.259,WIN
3,recall,casb_edited,79.791,smote,76.969,2.822,WIN


## Per-dataset matched averages (%)

gmean      f1  precision  recall
dataset                method                                         
abalone19              adasyn        45.663   4.566      2.657  32.202
                       bsmote        23.950   2.973      1.776  13.452
                       casb          44.312   5.364      4.073  31.071
                       casb_edited   49.666   4.726      2.624  36.071
                       casb_relaxed  47.640   5.339      3.296  32.262
                       casb_ros      41.533   5.295      7.173  30.357
                       original       0.000   0.000      0.000   0.000
                       ros           41.807   5.095      5.860  30.774
                       smote         45.046   4.437      2.553  31.488
abalone9-18            adasyn        76.879  48.229     40.010  65.729
                       bsmote        71.286  48.090     45.328  55.868
                       casb          74.996  51.664     47.291  61.528
                       casb_edited   77.422  46.979     37.672  66.944
                       casb_relaxed  75.868  50.249     44.256  63.507
                       casb_ros      74.181  51.866     48.700  60.590
                       original      40.672  33.895     72.083  25.000
                       ros           75.203  49.209     45.097  63.021
                       smote         74.337  45.900     38.448  61.875
ecoli-0_vs_1           adasyn        96.941  96.214     96.933  95.833
                       bsmote        97.388  97.047     98.655  95.677
                       casb          97.223  96.928     98.922  95.188
                       casb_edited   97.199  96.736     98.090  95.667
                       casb_relaxed  97.387  97.106     98.942  95.510
                       casb_ros      97.184  96.849     98.732  95.188
                       original      97.080  96.874     99.412  94.688
                       ros           97.124  96.891     99.256  94.854
                       smote         97.132  96.829     98.922  95.021
ecoli1                 adasyn        88.438  77.990     70.022  90.000
                       bsmote        88.004  78.021     71.167  88.208
                       casb          88.063  78.532     72.404  87.375
                       casb_edited   88.604  77.532     68.413  91.031
                       casb_relaxed  87.855  78.490     72.744  86.729
                       casb_ros      87.964  78.342     72.170  87.385
                       original      84.977  79.131     83.260  76.365
                       ros           87.425  78.121     72.909  85.594
                       smote         88.428  79.333     73.743  87.427
ecoli2                 adasyn        90.795  78.580     71.829  89.182
                       bsmote        89.721  78.072     72.621  86.841
                       casb          92.058  82.800     78.443  89.659
                       casb_edited   92.301  83.115     78.666  90.159
                       casb_relaxed  92.046  83.405     79.684  89.159
                       casb_ros      92.034  82.681     78.181  89.636
                       original      86.797  81.665     87.350  77.545
                       ros           91.346  82.198     78.539  88.227
                       smote         91.699  83.744     81.264  88.227
ecoli3                 adasyn        84.875  60.383     49.716  81.786
                       bsmote        85.169  62.388     52.984  81.071
                       casb          83.188  61.632     53.766  77.143
                       casb_edited   86.439  61.149     49.118  84.643
                       casb_relaxed  84.294  63.049     55.090  78.929
                       casb_ros      83.856  62.056     53.742  78.571
                       original      71.033  59.254     71.325  53.929
                       ros           83.547  60.788     52.066  78.214
                       smote         84.474  62.473     53.682  79.643
ecoli4                 adasyn        90.892  78.155     75.369

Results are metric-wise. No composite winner is reported. progress.md is unchanged.


## Statistical significance: independent classifier-specific analysis

Tests below read the independently reproduced tables in `statistical result/`. Classifiers remain separate. Each fixed variant is tested against all compatible baselines across four metrics, with Holm correction across 112 comparisons for SVM/KNN (including no resampling, RUS and SMOTE-ENN) and 80 for RF/LR. Pairwise comparisons use 44 complete datasets where possible, or 40 for ADASYN. Friedman and post hoc comparisons use strict complete cohorts.

Source-family means, lexicographically selected representative tasks, effect sizes and exact sign tests are included. Family-best maxima are descriptive and never tested as a deployable method. RF/LR selection was post hoc. The earlier mean tables retain their explicitly labelled original four-resampler comparison scope.


In [5]:
# Reproduce the source tables by running statistical result/statistical_analysis.py.
stat_path=Path('statistical result')
statistical_pairs=pd.read_csv(stat_path/'wilcoxon_holm_results.csv')
statistical_pairs=statistical_pairs[statistical_pairs.classifier.isin(['svm','knn','rf','lr'])]
source_family_pairs=statistical_pairs[statistical_pairs.analysis=='source_family_mean']
representative_pairs=statistical_pairs[statistical_pairs.analysis=='one_task_per_family']
primary_pairs=statistical_pairs[statistical_pairs.analysis=='dataset_pairs']
assert len(primary_pairs)==384
omnibus=pd.read_csv(stat_path/'friedman_results.csv');omnibus=omnibus[omnibus.classifier.isin(['svm','knn','rf','lr'])]
for title,t in [('Friedman tests',omnibus),('Fixed-variant paired tests',primary_pairs),('Source-family sensitivity',source_family_pairs),('One representative task per family',representative_pairs)]:
    display(Markdown('## '+title))
    with pd.option_context('display.max_rows',400,'display.max_columns',20):display(t)
    if 'direction' in t:display(t.groupby(['classifier','direction']).size().unstack(fill_value=0))
display(Markdown('## Retrospective family counts: both baseline populations'))
family_analysis=pd.read_csv(stat_path/'family_win_analysis.csv')
display(family_analysis[(family_analysis.row_type=='summary')&family_analysis.classifier.isin(['svm','knn','rf','lr'])])
print('No pooled-classifier primary test or significance test on family maxima. See the statistical report for population and multiplicity definitions.')


## Friedman tests

,classifier,metric,n_datasets,n_methods,statistic,p_raw,mean_ranks_json,dataset_units,methods,p_holm_four_metrics,significant
0,svm,gmean,40,11,88.466730,1.078023e-14,"{""adasyn"": 4.9125, ""bsmote"": 6.2125, ""casb"": 6...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,2.156046e-14,True
1,svm,f1,40,11,69.496386,5.545862e-11,"{""adasyn"": 6.225, ""bsmote"": 5.975, ""casb"": 4.9...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,5.545862e-11,True
2,svm,precision,40,11,104.548954,6.672511e-18,"{""adasyn"": 7.425, ""bsmote"": 6.4375, ""casb"": 4....",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,2.001753e-17,True
3,svm,recall,40,11,155.457454,2.802799e-28,"{""adasyn"": 4.375, ""bsmote"": 6.2125, ""casb"": 6....",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,1.121120e-27,True
4,knn,gmean,40,11,94.058905,8.362724e-16,"{""adasyn"": 4.4, ""bsmote"": 5.675, ""casb"": 5.55,...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,1.672545e-15,True
5,knn,f1,40,11,44.012885,3.274170e-06,"{""adasyn"": 6.75, ""bsmote"": 5.575, ""casb"": 4.35...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,3.274170e-06,True
6,knn,precision,40,11,121.653160,2.336348e-21,"{""adasyn"": 7.95, ""bsmote"": 6.4375, ""casb"": 4.3...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,7.009045e-21,True
7,knn,recall,40,11,174.372939,3.444562e-32,"{""adasyn"": 3.5375, ""bsmote"": 5.4875, ""casb"": 6...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;rus;smoteenn;...,1.377825e-31,True
8,rf,gmean,40,9,109.298434,5.306377e-20,"{""adasyn"": 3.3375, ""bsmote"": 5.275, ""casb"": 5....",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;casb;casb_rel...,1.591913e-19,True
9,rf,f1,40,9,51.126180,2.481214e-08,"{""adasyn"": 4.7625, ""bsmote"": 5.7, ""casb"": 4.47...",abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,original;smote;bsmote;ros;adasyn;casb;casb_rel...,2.481214e-08,True


## Fixed-variant paired tests

,analysis,classifier,metric,method,baseline,paired_units,n,casb_mean,baseline_mean,mean_difference,...,ties,losses,sign_p_raw,p_holm,sign_p_holm,correction_n,significant,direction,p_holm_core80,omnibus_significant
0,dataset_pairs,svm,gmean,casb,original,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.621535,0.193775,...,4,2,1.493390e-09,0.000005,1.612862e-07,112,True,CASB higher,0.000004,NaN
1,dataset_pairs,svm,gmean,casb,smote,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.825556,-0.010246,...,8,17,8.679394e-01,1.000000,1.000000e+00,112,False,not significant,1.000000,NaN
2,dataset_pairs,svm,gmean,casb,bsmote,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.807672,0.007639,...,9,19,7.358788e-01,1.000000,1.000000e+00,112,False,not significant,1.000000,NaN
3,dataset_pairs,svm,gmean,casb,ros,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.834260,-0.018949,...,7,22,3.240086e-01,0.610890,1.000000e+00,112,False,not significant,0.495795,NaN
4,dataset_pairs,svm,gmean,casb,adasyn,abalone19;abalone9-18;ecoli-0_vs_1;ecoli1;ecol...,40,0.804836,0.814989,-0.010153,...,4,24,6.524534e-02,1.000000,1.000000e+00,112,False,not significant,1.000000,NaN
5,dataset_pairs,svm,gmean,casb,rus,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.815760,-0.000449,...,2,22,8.776143e-01,1.000000,1.000000e+00,112,False,not significant,NaN,NaN
6,dataset_pairs,svm,gmean,casb,smoteenn,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.815311,0.839443,-0.024132,...,5,20,1.000000e+00,1.000000,1.000000e+00,112,False,not significant,NaN,NaN
7,dataset_pairs,svm,gmean,casb_relaxed,original,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.823414,0.621535,0.201878,...,4,2,1.493390e-09,0.000005,1.612862e-07,112,True,CASB higher,0.000004,NaN
8,dataset_pairs,svm,gmean,casb_relaxed,smote,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.823414,0.825556,-0.002143,...,7,16,5.113758e-01,1.000000,1.000000e+00,112,False,not significant,1.000000,NaN
9,dataset_pairs,svm,gmean,casb_relaxed,bsmote,abalone19;abalone9-18;ecoli-0-1-3-7_vs_2-6;eco...,44,0.823414,0.807672,0.015742,...,10,13,2.294810e-01,1.000000,1.000000e+00,112,False,not significant,1.000000,NaN


direction,CASB higher,CASB lower,not significant
classifier,,,
knn,18,13,81
lr,14,5,61
rf,21,4,55
svm,27,4,81


## Source-family sensitivity

... (4 outputs left out)

direction,not significant
classifier,
knn,112
lr,80
rf,80
svm,112


## Retrospective family counts: both baseline populations

,row_type,population,classifier,metric,dataset,dataset_family,best_variant,best_baseline,family_score,baseline_score,winning_margin,result,n,wins,ties,losses,win_percent,majority
2005,summary,all_available,knn,f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,14.0,2.0,24.0,35.0,False
2006,summary,all_available,knn,gmean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,11.0,1.0,28.0,27.5,False
2007,summary,all_available,knn,joint_gmean_f1_same_variant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,8.0,1.0,31.0,20.0,False
2008,summary,all_available,knn,precision,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,5.0,1.0,34.0,12.5,False
2009,summary,all_available,knn,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,5.0,4.0,31.0,12.5,False
2010,summary,all_available,lr,f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,15.0,3.0,22.0,37.5,False
2011,summary,all_available,lr,gmean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,15.0,2.0,23.0,37.5,False
2012,summary,all_available,lr,joint_gmean_f1_same_variant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,9.0,2.0,29.0,22.5,False
2013,summary,all_available,lr,precision,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,4.0,1.0,35.0,10.0,False
2014,summary,all_available,lr,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,16.0,7.0,17.0,40.0,False


No pooled-classifier primary test or significance test on family maxima. See the statistical report for population and multiplicity definitions.
